# jiuzhang-sdk 本地 GBS 能力示例

本 Notebook 演示不依赖云平台的本地能力：GBS 数学计算、本地采样、样本分布统计，以及本地 GBS Program 的 JSON / 本地程序文本 / XIR 序列化。

这些能力用于教学、算法原型、Mock 数据和本地 sanity check；真实云平台任务仍使用 `CloudClient`。

## 1. 导入本地能力

In [1]:
import jiuzhang
import numpy as np

from jiuzhang.local.gbs import (
    GBSProgram,
    dumps_ir,
    hafnian,
    random_adjacency_matrix,
    sample_gbs,
    samples_to_distribution,
    to_blackbird,
    to_xir,
    torontonian,
)

print("jiuzhang-sdk version:", jiuzhang.__version__)

jiuzhang-sdk version: 0.1.0a30


## 2. GBS 数学计算

`hafnian` 和 `torontonian` 是本地薄封装：用户使用九章 SDK 的方法名和参数风格，内部调用本地数学后端。

参数说明：

- `matrix` / `adjacency`：方阵，表示图邻接矩阵或待计算矩阵。
- `recursive`：`torontonian` 的后端递归计算开关。

In [9]:
adjacency = np.array([
    [0.0, 1.0],
    [1.0, 0.0],
])

tor_matrix = np.array([
    [0.10, 0.02],
    [0.02, 0.10],
])

hafnian_value = hafnian(adjacency)
torontonian_value = torontonian(tor_matrix)

print("Hafnian(adjacency):", hafnian_value)
print("Torontonian(tor_matrix):", torontonian_value)

Hafnian(adjacency): 1.0
Torontonian(tor_matrix): 0.11138556118596776


## 3. 生成本地 GBS 样本

首次运行采样函数时，本地后端可能会进行编译，第一次会稍慢；后续调用会明显变快。

参数说明：

- `modes`：模式数，决定邻接矩阵维度；示例使用 `8 x 8`。
- `scale`：随机边权的缩放系数。
- `shots`：采样次数。
- `mean_photon_count`：目标平均光子数。
- `detector`：探测器模型，`pnr` 表示光子数分辨探测，`threshold` 表示阈值探测。
- `cutoff`：单个模式的光子数截断，只影响 `pnr`。
- `max_photons`：总光子数上限。
- `seed`：随机种子，用于复现实验。

In [10]:
# modes: 本地 GBS 图的模式数；矩阵维度为 modes x modes。
# scale: 随机边权缩放系数，数值越大通常对应更强的边权。
graph = random_adjacency_matrix(8, scale=0.16, seed=7)

print("Adjacency matrix shape:", graph.shape)
print(np.round(graph, 4))

# shots: 采样次数；mean_photon_count: 目标平均光子数。
# detector="pnr": 光子数分辨探测；cutoff: 单模式光子数截断。
# max_photons: 总光子数上限；seed: 让本地采样结果可复现。
samples = sample_gbs(
    graph,
    shots=24,
    mean_photon_count=1.0,
    detector="pnr",
    cutoff=4,
    max_photons=12,
    seed=123,
)

print("Local samples preview:")
for row in samples[:10]:
    print(row)

print("Total samples:", len(samples))

Adjacency matrix shape: (8, 8)
[[0.     0.1436 0.1241 0.036  0.048  0.1398 0.0008 0.1314]
 [0.1436 0.     0.0485 0.0445 0.0408 0.0712 0.0807 0.0886]
 [0.1241 0.0485 0.     0.1582 0.0344 0.0256 0.098  0.007 ]
 [0.036  0.0445 0.1582 0.     0.1007 0.0823 0.0795 0.0396]
 [0.048  0.0408 0.0344 0.1007 0.     0.0006 0.1328 0.0247]
 [0.1398 0.0712 0.0256 0.0823 0.0006 0.     0.0146 0.0866]
 [0.0008 0.0807 0.098  0.0795 0.1328 0.0146 0.     0.024 ]
 [0.1314 0.0886 0.007  0.0396 0.0247 0.0866 0.024  0.    ]]
Local samples preview:
[0, 2, 0, 0, 0, 2, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[1, 0, 1, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 1, 1, 0, 0, 0]
[0, 0, 1, 0, 1, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0]
[1, 0, 1, 0, 0, 0, 0, 0]
Total samples: 24


## 4. 样本分布统计

`sample_gbs` 返回的是二维整数样本，`samples_to_distribution` 可以把样本转换成归一化后的 pattern 分布。

参数说明：

- `samples`：二维数组，每一行是一条采样结果，每一列对应一个模式。

In [11]:
distribution = samples_to_distribution(samples)

print("Pattern distribution:")
for pattern, probability in sorted(distribution.items(), key=lambda item: item[1], reverse=True):
    print(pattern, f"{probability:.3f}")

print("Probability sum:", sum(distribution.values()))

Pattern distribution:
(0, 0, 0, 0, 0, 0, 0, 0) 0.583
(1, 0, 1, 0, 0, 0, 0, 0) 0.083
(0, 2, 0, 0, 0, 2, 0, 0) 0.042
(0, 0, 0, 1, 1, 0, 0, 0) 0.042
(0, 0, 1, 0, 1, 0, 0, 0) 0.042
(2, 0, 1, 0, 0, 0, 0, 1) 0.042
(1, 0, 1, 2, 0, 0, 0, 0) 0.042
(0, 0, 1, 1, 0, 0, 0, 0) 0.042
(1, 0, 0, 0, 0, 0, 0, 1) 0.042
(0, 0, 0, 0, 1, 0, 1, 0) 0.042
Probability sum: 1.0


## 5. 构造本地 GBS Program

`GBSProgram` 用于描述本地 GBS 程序，并可序列化为 JSON / 本地程序文本 / XIR。

参数说明：

- `modes`：程序包含的模式数。
- `squeezing([...])`：每个模式的压缩强度，列表长度必须等于 `modes`。
- `edge(mode_a, mode_b, weight)`：添加一条无向加权边。
- `measure_fock(shots=...)`：添加光子数分辨测量，`shots` 表示测量次数。

In [12]:
# modes: 程序模式数；squeezing_values: 每个模式的压缩强度。
squeezing_values = [0.35, 0.32, 0.30, 0.28, 0.25, 0.22, 0.20, 0.18]

program = (
    GBSProgram(modes=8)
    .squeezing(squeezing_values)
    .edge(0, 1, 0.15)
    .edge(1, 2, 0.10)
    .edge(2, 3, 0.12)
    .edge(4, 5, 0.11)
    .edge(5, 6, 0.09)
    .edge(6, 7, 0.13)
    .measure_fock(shots=240)
)

program_ir = program.to_dict()
print(program_ir)

{'schema': 'jiuzhang.local.gbs.v1', 'name': 'gbs_program', 'modes': 8, 'operations': [{'name': 'squeezing', 'params': {'r': 0.35, 'phi': 0.0}, 'modes': [0]}, {'name': 'squeezing', 'params': {'r': 0.32, 'phi': 0.0}, 'modes': [1]}, {'name': 'squeezing', 'params': {'r': 0.3, 'phi': 0.0}, 'modes': [2]}, {'name': 'squeezing', 'params': {'r': 0.28, 'phi': 0.0}, 'modes': [3]}, {'name': 'squeezing', 'params': {'r': 0.25, 'phi': 0.0}, 'modes': [4]}, {'name': 'squeezing', 'params': {'r': 0.22, 'phi': 0.0}, 'modes': [5]}, {'name': 'squeezing', 'params': {'r': 0.2, 'phi': 0.0}, 'modes': [6]}, {'name': 'squeezing', 'params': {'r': 0.18, 'phi': 0.0}, 'modes': [7]}, {'name': 'edge', 'params': {'weight': 0.15}, 'modes': [0, 1]}, {'name': 'edge', 'params': {'weight': 0.1}, 'modes': [1, 2]}, {'name': 'edge', 'params': {'weight': 0.12}, 'modes': [2, 3]}, {'name': 'edge', 'params': {'weight': 0.11}, 'modes': [4, 5]}, {'name': 'edge', 'params': {'weight': 0.09}, 'modes': [5, 6]}, {'name': 'edge', 'params':

## 6. 序列化为 JSON / 本地程序文本 / XIR

公开 API 只暴露九章本地 Program；第三方 IR 后端只作为可选序列化实现。

In [ ]:
json_ir = dumps_ir(program)
blackbird_ir = to_blackbird(program)
xir_ir = to_xir(program)

print("JSON IR preview:")
print("\n".join(json_ir.splitlines()[:12]))

print("\n本地程序文本 preview:")
print("\n".join(blackbird_ir.splitlines()[:8]))

print("\nXIR preview:")
print("\n".join(xir_ir.splitlines()[:8]))

## 7. 本地能力边界

本地 GBS 能力适合用于数学验证、示例教学、算法原型、Mock 数据和结果 sanity check。生产级远程任务仍应通过九章云平台提交。